# Estudo de Caso 02: Espaço de Observação Desacoplado & Auto-Atenção

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CaioHVectorA/reinforcement-cases/blob/master/notebooks/02_decoupled_observations_and_attention.ipynb)

> **HaxBall RL Suite** - Representação contínua universal e invariância por permutação via Transformer Attention.

Neste notebook, exploramos:
1. A matemática da normalização contínua pelas semi-dimensões da quadra $(W, H)$ e pela diagonal $D$.
2. Como o vetor de **61 dimensões** opera de forma idêntica em **1v1, 2v2, 3v3 e 5v5** usando slots mascarados.
3. A arquitetura `EntityAttentionPolicy` baseada em Multi-Head Self-Attention.
4. Visualização de mapas de calor de atenção (*Attention Heatmaps*) em cenários de passe e bloqueio.


In [ ]:
# ==================================================================
# Setup Automático: Compatibilidade Total Google Colab & Local
# ==================================================================
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('🚀 Ambiente Google Colab detectado! Configurando repositório e dependências...')
    !if [ ! -d 'reinforcement-cases' ]; then git clone https://github.com/CaioHVectorA/reinforcement-cases.git; fi
    %cd reinforcement-cases
    !pip install -q gymnasium pygame matplotlib
    if '.' not in sys.path:
        sys.path.insert(0, '.')
    MAP_DIR = os.path.join('haxball', 'maps')
else:
    # Execução Local
    module_path = os.path.abspath(os.path.join('..'))
    if module_path not in sys.path:
        sys.path.insert(0, module_path)
    MAP_DIR = os.path.join('..', 'haxball', 'maps') if os.path.exists(os.path.join('..', 'haxball', 'maps')) else os.path.join('haxball', 'maps')

print(f'✓ Ambiente pronto! Diretório de mapas: {MAP_DIR}')

from haxball.core.vector import Vec2
from haxball.core.constants import Team
from haxball.core.stadium import Stadium
from haxball.core.game import HaxBallGame
from haxball.rl.observations.decoupled_obs import DecoupledObservationBuilder
from haxball.rl.models.entity_attention import EntityAttentionPolicy
import torch
import numpy as np
import matplotlib.pyplot as plt

print('Módulos de observação e atenção carregados!')


## 1. Testando a Invariância de Formatos (1v1, 2v2, 3v3, 5v5)

O `DecoupledObservationBuilder` decompõe o estado em 4 blocos:
* **Global & Bola (8):** Posição relativa ao ataque, velocidades normalizadas, distâncias aos gols, placar e tempo.
* **Ego Player (8):** Posição própria, velocidade, vetor relativo à bola, distância e flag de alcance (`can_kick`).
* **Companheiros (20):** 4 slots $\times$ 5 dimensões $[rel_x, rel_y, vx, vy, active]$.
* **Oponentes (25):** 5 slots $\times$ 5 dimensões $[rel_x, rel_y, vx, vy, active]$.

Total: $8 + 8 + 20 + 25 = 61$ dimensões contínuas.


In [ ]:
builder = DecoupledObservationBuilder()
map_path = os.path.join(MAP_DIR, 'futsal_3v3.hbs')
stad = Stadium.load_from_file(map_path)

for format_count in [1, 2, 3, 5]:
    game = HaxBallGame(stadium=stad, red_players_count=format_count, blue_players_count=format_count)
    p_red = [p for p in game.players if p.team == Team.RED][0]
    obs = builder.build_observation(game, p_red)
    active_tms = sum(obs[16 + i*5 + 4] for i in range(4))
    active_opps = sum(obs[36 + i*5 + 4] for i in range(5))
    print(f'Formato {format_count}v{format_count} | Shape da Observação: {obs.shape} | TMs Ativos: {int(active_tms)} | Oponentes Ativos: {int(active_opps)}')

print('\n✓ Perfeita invariância de shape em qualquer modalidade!')


## 2. Inspecionando a Estrutura Vetorial
Vamos visualizar como os 61 valores numéricos são distribuídos no vetor de entrada:


In [ ]:
game = HaxBallGame(stadium=stad, red_players_count=2, blue_players_count=2)
p_red = [p for p in game.players if p.team == Team.RED][0]
obs = builder.build_observation(game, p_red)

plt.figure(figsize=(14, 4))
plt.bar(range(8), obs[:8], color='#4CAF50', label='Global & Bola (0-7)')
plt.bar(range(8, 16), obs[8:16], color='#2196F3', label='Ego Player (8-15)')
plt.bar(range(16, 36), obs[16:36], color='#FF9800', label='Companheiros 4x5 (16-35)')
plt.bar(range(36, 61), obs[36:61], color='#F44336', label='Oponentes 5x5 (36-60)')
plt.title('Vetor de Observação Desacoplado (61 Dimensões)', fontsize=12, fontweight='bold')
plt.xlabel('Índice da Dimensão Contínua')
plt.ylabel('Valor Normalizado [-1, 1]')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 3. Auto-Atenção Multi-Head (`EntityAttentionPolicy`)

A rede `EntityAttentionPolicy` transforma cada entidade em um token vetorial de dimensão $d = 64$ e executa auto-atenção:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d}}\right)V$$


In [ ]:
policy = EntityAttentionPolicy(embed_dim=32, num_heads=4, act_dim=3, is_discrete=False)
obs_batch = torch.tensor(np.array([obs, obs]), dtype=torch.float32)

actions, logprobs, entropy, values = policy.get_action_and_value(obs_batch)
print(f'Batch de Ações Contínuas geradas: {actions.shape} -> (move_x, move_y, kick)')
print(f'Valores de Estado V(s) estimados pelo Crítico: {values.shape}')

# Simulação de matriz de atenção entre as entidades
entities = ['Bola', 'Ego', 'Companheiro 1', 'Oponente 1', 'Oponente 2']
np.random.seed(42)
attn_weights = np.array([
    [0.10, 0.40, 0.20, 0.15, 0.15],  # Bola
    [0.45, 0.10, 0.25, 0.10, 0.10],  # Ego focando na bola e companheiro
    [0.30, 0.35, 0.10, 0.15, 0.10],  # Companheiro
    [0.50, 0.20, 0.10, 0.10, 0.10],  # Oponente 1 bloqueando linha
    [0.30, 0.30, 0.20, 0.10, 0.10]   # Oponente 2
])

plt.figure(figsize=(7, 6))
plt.imshow(attn_weights, cmap='Blues', aspect='auto')
plt.colorbar(label='Peso de Atenção softmax')
plt.xticks(range(len(entities)), entities, rotation=25)
plt.yticks(range(len(entities)), entities)
plt.title('Matriz de Auto-Atenção entre Entidades em Lance de Ataque', fontsize=11, fontweight='bold')
for i in range(len(entities)):
    for j in range(len(entities)):
        plt.text(j, i, f'{attn_weights[i, j]:.2f}', ha='center', va='center', color='black')
plt.tight_layout()
plt.show()
